In [0]:
from pyspark.sql.functions import col, sum as spark_sum

In [0]:
# Configure ADLS access

spark.conf.set(
    "fs.azure.account.key.@storageaccount.dfs.core.windows.net",
    "Access token"
)

In [0]:
# ============================================================
# SMART AGRICULTURE - BRONZE INGESTION + DATA VALIDATION
# ============================================================


# ============================================================
# 1. WEATHER
# ============================================================

weather_raw_path = "abfss://raw@smartagrinil.dfs.core.windows.net/weather/weather.csv"
weather_bronze_path = "abfss://processed@smartagrinil.dfs.core.windows.net/bronze/weather"

weather_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(weather_raw_path)
)

print("========== WEATHER VALIDATION ==========")

# Row count
weather_count = weather_df.count()
print("Rows:", weather_count)

# Check empty
if weather_count == 0:
    raise Exception("Weather dataset is empty!")

# Columns
print("Columns:", weather_df.columns)

# Required columns
weather_required = [
    "date",
    "latitude",
    "longitude",
    "district",
    "temperature_c"
]

missing = [c for c in weather_required if c not in weather_df.columns]

if missing:
    raise Exception(f"Weather missing columns: {missing}")

print("Required columns: PASS")

# Null check
weather_df.select(
    [spark_sum(col(c).isNull().cast("int")).alias(c)
     for c in weather_required]
).show()

# Duplicate check
weather_duplicates = weather_count - weather_df.dropDuplicates().count()
print("Duplicate rows:", weather_duplicates)

# Numeric validation
if "temperature_c" in weather_df.columns:
    invalid_temp = weather_df.filter(
        col("temperature_c").isNotNull() &
        ((col("temperature_c") < -50) | (col("temperature_c") > 60))
    ).count()

    print("Invalid temperature rows:", invalid_temp)

# Write Bronze
weather_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(weather_bronze_path)

print("Weather Bronze: SUCCESS")


# ============================================================
# 2. SOIL
# ============================================================

soil_raw_path = "abfss://raw@smartagrinil.dfs.core.windows.net/soil/soil.csv"
soil_bronze_path = "abfss://processed@smartagrinil.dfs.core.windows.net/bronze/soil"

soil_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(soil_raw_path)
)

print("\n========== SOIL VALIDATION ==========")

soil_count = soil_df.count()
print("Rows:", soil_count)

if soil_count == 0:
    raise Exception("Soil dataset is empty!")

print("Columns:", soil_df.columns)

# Display schema so we know actual column names
soil_df.printSchema()

# Null check for all columns
soil_df.select(
    [spark_sum(col(c).isNull().cast("int")).alias(c)
     for c in soil_df.columns]
).show()

# Duplicate check
soil_duplicates = soil_count - soil_df.dropDuplicates().count()
print("Duplicate rows:", soil_duplicates)

# Write Bronze
soil_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(soil_bronze_path)

print("Soil Bronze: SUCCESS")


# ============================================================
# 3. CROP YIELD
# ============================================================

crop_raw_path = "abfss://raw@smartagrinil.dfs.core.windows.net/crop_yield/crop_yield.csv"
crop_bronze_path = "abfss://processed@smartagrinil.dfs.core.windows.net/bronze/crop_yield"

crop_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(crop_raw_path)
)

print("\n========== CROP YIELD VALIDATION ==========")

crop_count = crop_df.count()
print("Rows:", crop_count)

if crop_count == 0:
    raise Exception("Crop Yield dataset is empty!")

print("Columns:", crop_df.columns)

crop_df.printSchema()

# Null check
crop_df.select(
    [spark_sum(col(c).isNull().cast("int")).alias(c)
     for c in crop_df.columns]
).show()

# Duplicate check
crop_duplicates = crop_count - crop_df.dropDuplicates().count()
print("Duplicate rows:", crop_duplicates)

# Check negative numeric values
numeric_columns = [
    field.name
    for field in crop_df.schema.fields
    if str(field.dataType) in ["IntegerType", "LongType", "DoubleType", "FloatType"]
]

for c in numeric_columns:
    invalid = crop_df.filter(col(c) < 0).count()
    print(f"{c} negative rows:", invalid)

# Write Bronze
crop_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(crop_bronze_path)

print("Crop Yield Bronze: SUCCESS")


# ============================================================
# 4. MARKET PRICE
# ============================================================

market_raw_path = "abfss://raw@smartagrinil.dfs.core.windows.net/market_price/market_price.csv"
market_bronze_path = "abfss://processed@smartagrinil.dfs.core.windows.net/bronze/market_price"

market_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(market_raw_path)
)

print("\n========== MARKET PRICE VALIDATION ==========")

market_count = market_df.count()
print("Rows:", market_count)

if market_count == 0:
    raise Exception("Market Price dataset is empty!")

print("Columns:", market_df.columns)

market_df.printSchema()

# Null check
market_df.select(
    [spark_sum(col(c).isNull().cast("int")).alias(c)
     for c in market_df.columns]
).show()

# Duplicate check
market_duplicates = market_count - market_df.dropDuplicates().count()
print("Duplicate rows:", market_duplicates)

# Check negative numeric values
numeric_columns = [
    field.name
    for field in market_df.schema.fields
    if str(field.dataType) in ["IntegerType", "LongType", "DoubleType", "FloatType"]
]

for c in numeric_columns:
    invalid = market_df.filter(col(c) < 0).count()
    print(f"{c} negative rows:", invalid)

# Write Bronze
market_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(market_bronze_path)

print("Market Price Bronze: SUCCESS")


# ============================================================
# FINAL
# ============================================================

print("\n========================================")
print("ALL 4 DATASETS VALIDATED AND INGESTED")
print("========================================")

========== WEATHER VALIDATION ==========
Rows: 1520
Columns: ['date', 'district', 'latitude', 'longitude', 'temperature_c', 'precipitation_mm', 'relative_humidity_pct', 'solar_radiation_kwh_m2']
Required columns: PASS
+----+--------+---------+--------+-------------+
|date|latitude|longitude|district|temperature_c|
+----+--------+---------+--------+-------------+
|   0|       0|        0|       0|           20|
+----+--------+---------+--------+-------------+

Duplicate rows: 20
Invalid temperature rows: 0
Weather Bronze: SUCCESS

========== SOIL VALIDATION ==========
Rows: 1520
Columns: ['sample_date', 'state', 'district', 'soil_type', 'ph', 'organic_carbon_pct', 'nitrogen_kg_ha', 'phosphorus_kg_ha', 'potassium_kg_ha', 'moisture_pct']
root
 |-- sample_date: date (nullable = true)
 |-- state: string (nullable = true)
 |-- district: string (nullable = true)
 |-- soil_type: string (nullable = true)
 |-- ph: double (nullable = true)
 |-- organic_carbon_pct: double (nullable = true)
 |-- ni